In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ---------- paths ----------
ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed_data"
RESULTS_DIR = ROOT / "results" / "intermediate"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DONOR_FILE = PROCESSED_DIR / "matching_weighted_ks.csv"
DETAIL_FILE = RESULTS_DIR / "Table2_TF_ANN_detail.csv"
SUMMARY_FILE = RESULTS_DIR / "Table2_TF_ANN_summary.csv"


# ---------- config ----------
TARGET = "power"
BASE_FEATURES = [
    "wind_speed",
    "temperature",
    "turbulence_intensity",
    "std_wind_direction",
]
ANGLE_FEATURE = "wind_direction"

K = 7
SEED = 2026
TRAIN_YEAR = 2017
TEST_YEARS = [2017, 2018]

# ANN: 8-16-8
HIDDEN_LAYERS = (8, 16, 8)
LEARNING_RATE = 1e-3
BATCH_SIZE = 64
EPOCHS = 100


# ---------- utils ----------
def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def load_turbine_year(turbine_id: int, year: int) -> pd.DataFrame:
    path = DATA_DIR / f"Turbine{int(turbine_id)}_{int(year)}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    df = pd.read_csv(path)
    required = BASE_FEATURES + [ANGLE_FEATURE, TARGET]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {path}: {missing}")

    df = df[required].copy()
    for c in required:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna().copy()
    if df.empty:
        raise ValueError(f"No usable rows in {path}")

    rad = np.deg2rad(df[ANGLE_FEATURE].to_numpy())
    df["wind_direction_sin"] = np.sin(rad)
    df["wind_direction_cos"] = np.cos(rad)
    df = df.drop(columns=[ANGLE_FEATURE])

    return df


def feature_names():
    return BASE_FEATURES + ["wind_direction_sin", "wind_direction_cos"]


def read_donor_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing donor file: {path}")

    df = pd.read_csv(path)
    if "target" not in df.columns:
        df = df.rename(columns={df.columns[0]: "target"})

    donor_cols = [c for c in df.columns if c.startswith("donor")]
    if not donor_cols:
        raise ValueError(f"No donor columns found in {path}")

    donor_cols = sorted(donor_cols, key=lambda x: int(x.replace("donor", "")))
    keep = ["target"] + donor_cols
    df = df[keep].copy()
    df["target"] = pd.to_numeric(df["target"], errors="coerce")
    df = df.dropna(subset=["target"]).copy()
    df["target"] = df["target"].astype(int)

    for c in donor_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


def build_ann():
    return Pipeline(
        steps=[
            ("x_scaler", StandardScaler()),
            (
                "ann",
                MLPRegressor(
                    hidden_layer_sizes=HIDDEN_LAYERS,
                    activation="relu",
                    solver="adam",
                    learning_rate_init=LEARNING_RATE,
                    batch_size=BATCH_SIZE,
                    max_iter=EPOCHS,
                    random_state=SEED,
                    shuffle=True,
                    early_stopping=False,
                ),
            ),
        ]
    )


def fit_one_donor(donor_id: int, target_id: int, test_year: int):
    feats = feature_names()

    train_df = load_turbine_year(donor_id, TRAIN_YEAR)
    test_df = load_turbine_year(target_id, test_year)

    x_train = train_df[feats].to_numpy()
    y_train = train_df[TARGET].to_numpy()
    x_test = test_df[feats].to_numpy()
    y_test = test_df[TARGET].to_numpy()

    model = build_ann()

    start = time.time()
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    runtime = time.time() - start

    return {
        "pred": pred,
        "actual": y_test,
        "rmse": rmse(y_test, pred),
        "mae": mae(y_test, pred),
        "runtime": runtime,
    }


def run():
    donor_df = read_donor_table(DONOR_FILE)
    donor_cols = [c for c in donor_df.columns if c.startswith("donor")]
    targets = sorted(donor_df["target"].unique())

    detail_rows = []

    for target_id in targets:
        row = donor_df.loc[donor_df["target"] == target_id]
        if row.empty:
            continue

        donors = row[donor_cols].iloc[0].dropna().astype(int).tolist()
        donors = [d for d in donors if d != target_id]
        donors = donors[:K]
        if not donors:
            continue

        for test_year in TEST_YEARS:
            donor_preds = []
            donor_runtimes = []
            donor_rmses = []
            donor_maes = []
            donors_used = []
            actual = None

            for donor_id in donors:
                try:
                    res = fit_one_donor(donor_id, target_id, test_year)
                except Exception:
                    continue

                donor_preds.append(res["pred"])
                donor_runtimes.append(res["runtime"])
                donor_rmses.append(res["rmse"])
                donor_maes.append(res["mae"])
                donors_used.append(donor_id)
                actual = res["actual"]

            if not donor_preds:
                continue

            ensemble_pred = np.mean(np.vstack(donor_preds), axis=0)

            detail_rows.append(
                {
                    "method": "TF_ANN",
                    "target": target_id,
                    "year": test_year,
                    "donors_used": ",".join(map(str, donors_used)),
                    "n_models": len(donor_preds),
                    "rmse": rmse(actual, ensemble_pred),
                    "mae": mae(actual, ensemble_pred),
                    "runtime_sec": float(np.sum(donor_runtimes)),
                    "mean_single_rmse": float(np.mean(donor_rmses)),
                    "mean_single_mae": float(np.mean(donor_maes)),
                }
            )

    detail_df = pd.DataFrame(detail_rows)
    detail_df.to_csv(DETAIL_FILE, index=False)

    if detail_df.empty:
        summary_df = pd.DataFrame(
            columns=["method", "year", "avg_rmse", "avg_mae", "total_runtime_sec"]
        )
    else:
        summary_df = (
            detail_df.groupby(["method", "year"], as_index=False)
            .agg(
                avg_rmse=("rmse", "mean"),
                avg_mae=("mae", "mean"),
                total_runtime_sec=("runtime_sec", "sum"),
            )
        )

    summary_df.to_csv(SUMMARY_FILE, index=False)


if __name__ == "__main__":
    run()